# 01 - Panel DAGMA/SISAIRE

Objetivo: preparar la verdad terreno de Situación 3.

En palabras simples: antes de hacer mapas o modelos, revisamos que las estaciones reales tengan fechas, coordenadas, contaminantes y valores suficientes. Luego agregamos los datos horarios a una tabla diaria, porque los horizontes del proyecto son `T+1`, `T+3` y `T+7` días.

## Preguntas De Control

1. ¿Qué estación midió el dato?
2. ¿Qué contaminante midió?
3. ¿Cuándo lo midió?
4. ¿Dónde está la estación?
5. ¿El valor parece usable para validar un mapa?

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

In [2]:
ROOT = Path('/workspace/geovision-cali-hf')
INPUT_PATH = ROOT / 'data/DAGMA/processed/panel_dagma_sisaire_largo_2020_2024.parquet'
OUT_DIR = ROOT / 'outputs/situacion3/01_panel_dagma'
OUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUT_DIR

(PosixPath('/workspace/geovision-cali-hf/data/DAGMA/processed/panel_dagma_sisaire_largo_2020_2024.parquet'),
 PosixPath('/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma'))

## 1. Cargar Panel Largo

Usamos el formato largo porque cada fila dice: estación, contaminante, fecha, valor y coordenadas. Eso es cómodo para LOO-CV por contaminante.

In [3]:
df = pd.read_parquet(INPUT_PATH)
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
df['date'] = pd.to_datetime(df['date']).dt.date
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')

# Algunas fuentes Excel tienen valores válidos pero no traen coordenadas.
# Rellenamos lat/lon con las coordenadas conocidas de la misma estación.
# Normalizamos nombres para empatar variantes como 'Transitoria - Navarro' y 'TRANSITORIA-NAVARRO'.
def normalize_station_name(series):
    return (
        series.fillna('')
        .astype(str)
        .str.upper()
        .str.replace(r'[^A-Z0-9]+', ' ', regex=True)
        .str.strip()
    )

df['_station_key'] = normalize_station_name(df['estacion'])
df['estacion_original'] = df['estacion']
df['estacion'] = df['_station_key'].str.title()
station_coords_ref = (
    df.dropna(subset=['lat', 'lon'])
    .groupby('_station_key', dropna=False)[['lat', 'lon']]
    .median()
    .rename(columns={'lat': 'lat_ref', 'lon': 'lon_ref'})
    .reset_index()
)
df = df.merge(station_coords_ref, on='_station_key', how='left')
coord_missing_before = int(df[['lat', 'lon']].isna().any(axis=1).sum())
df['lat'] = df['lat'].fillna(df['lat_ref'])
df['lon'] = df['lon'].fillna(df['lon_ref'])
coord_missing_after = int(df[['lat', 'lon']].isna().any(axis=1).sum())
coord_fill_summary = {
    'coord_missing_before_fill': coord_missing_before,
    'coord_missing_after_fill': coord_missing_after,
    'stations_with_coord_reference': int(station_coords_ref['_station_key'].nunique()),
}
df = df.drop(columns=['_station_key', 'lat_ref', 'lon_ref'])

print('shape:', df.shape)
print(coord_fill_summary)
display(df.head())

shape: (803146, 15)
{'coord_missing_before_fill': 614204, 'coord_missing_after_fill': 0, 'stations_with_coord_reference': 9}


,fecha_hora,municipio,estacion,contaminante,unidad,valor,source_dataset,lat,lon,date,year,month,day,hour,estacion_original
0,2020-01-01,Santiago de Cali,Canaveralejo,NO2,ug/m3,NaN,api_panel,3.416366,-76.549613,2020-01-01,2020,1,1,0,Canaveralejo
1,2020-01-01,Santiago de Cali,Canaveralejo,O3,ug/m3,NaN,api_panel,3.416366,-76.549613,2020-01-01,2020,1,1,0,Canaveralejo
2,2020-01-01,Santiago de Cali,Canaveralejo,SO2,ug/m3,0.05,api_panel,3.416366,-76.549613,2020-01-01,2020,1,1,0,Canaveralejo
3,2020-01-01,Santiago de Cali,Compartir,NO2,ug/m3,NaN,api_panel,3.428260,-76.466584,2020-01-01,2020,1,1,0,Compartir
4,2020-01-01,Santiago de Cali,Compartir,O3,ug/m3,10.06,api_panel,3.428260,-76.466584,2020-01-01,2020,1,1,0,Compartir


In [4]:
required_cols = ['fecha_hora', 'municipio', 'estacion', 'contaminante', 'unidad', 'valor', 'lat', 'lon', 'date', 'year', 'month', 'day', 'hour']
missing_cols = sorted(set(required_cols) - set(df.columns))
assert not missing_cols, f'Faltan columnas requeridas: {missing_cols}'

summary = {
    'rows': int(len(df)),
    'columns': int(df.shape[1]),
    'min_fecha_hora': str(df['fecha_hora'].min()),
    'max_fecha_hora': str(df['fecha_hora'].max()),
    'n_estaciones': int(df['estacion'].nunique()),
    'n_contaminantes': int(df['contaminante'].nunique()),
    'contaminantes': sorted(df['contaminante'].dropna().astype(str).unique().tolist()),
}
summary

{'rows': 803146,
 'columns': 15,
 'min_fecha_hora': '2020-01-01 00:00:00',
 'max_fecha_hora': '2024-12-31 23:00:00',
 'n_estaciones': 9,
 'n_contaminantes': 3,
 'contaminantes': ['NO2', 'O3', 'SO2']}

## 2. Inventario Básico

Aquí revisamos cobertura por contaminante, estación y unidad. Si una estación no tiene coordenadas o un contaminante viene con unidades mezcladas, eso debe aparecer antes de modelar.

In [5]:
contaminant_summary = (
    df.groupby(['contaminante', 'unidad'], dropna=False)
    .agg(
        n_obs=('valor', 'size'),
        n_valid=('valor', 'count'),
        n_estaciones=('estacion', 'nunique'),
        min_fecha=('fecha_hora', 'min'),
        max_fecha=('fecha_hora', 'max'),
        min_valor=('valor', 'min'),
        p50_valor=('valor', 'median'),
        max_valor=('valor', 'max'),
    )
    .reset_index()
    .sort_values(['contaminante', 'unidad'])
)
display(contaminant_summary)

,contaminante,unidad,n_obs,n_valid,n_estaciones,min_fecha,max_fecha,min_valor,p50_valor,max_valor
0,NO2,ug/m3,178340,26259,9,2020-01-01 00:00:00,2024-12-31 23:00:00,0.0000,12.147438,91.857945
1,O3,ug/m3,339823,191010,9,2020-01-01 00:00:00,2024-12-31 23:00:00,0.0000,14.890284,290.409591
2,O3,NaN,247,247,2,2020-01-01 00:00:00,2020-12-11 12:00:00,4.5907,56.814600,179.016900
3,SO2,ug/m3,284637,132316,9,2020-01-01 00:00:00,2024-12-31 23:00:00,0.0000,2.853954,841.968850
4,SO2,NaN,99,99,1,2020-01-08 14:00:00,2020-12-11 12:00:00,4.7653,14.479200,32.964500


In [6]:
station_summary = (
    df.groupby(['municipio', 'estacion'], dropna=False)
    .agg(
        n_obs=('valor', 'size'),
        n_valid=('valor', 'count'),
        n_contaminantes=('contaminante', 'nunique'),
        lat=('lat', 'median'),
        lon=('lon', 'median'),
        min_fecha=('fecha_hora', 'min'),
        max_fecha=('fecha_hora', 'max'),
    )
    .reset_index()
    .sort_values(['municipio', 'estacion'])
)
display(station_summary.head(30))
print('estaciones sin coordenadas:', int(station_summary[['lat', 'lon']].isna().any(axis=1).sum()))

,municipio,estacion,n_obs,n_valid,n_contaminantes,lat,lon,min_fecha,max_fecha
0,Cali,Base Aerea,87694,52160,2,3.457128,-76.502303,2020-01-01 01:00:00,2024-12-31 23:00:00
1,Cali,Canaveralejo,43847,32922,1,3.416366,-76.549613,2020-01-01 01:00:00,2024-12-31 23:00:00
2,Cali,Compartir,43847,35055,1,3.428260,-76.466584,2020-01-01 01:00:00,2024-12-31 23:00:00
3,Cali,Era Obrero,43847,26534,1,3.457317,-76.506539,2020-01-01 01:00:00,2024-12-31 23:00:00
4,Cali,Ermita,43847,36919,1,3.455514,-76.530978,2020-01-01 01:00:00,2024-12-31 23:00:00
5,Cali,Flora,131541,48716,3,3.488218,-76.518058,2020-01-01 01:00:00,2024-12-31 23:00:00
6,Cali,Pance,43847,36339,1,3.304517,-76.531252,2020-01-01 01:00:00,2024-12-31 23:00:00
7,Cali,Transitoria Navarro,88039,13619,2,3.417183,-76.494960,2020-01-01 01:00:00,2024-12-31 23:00:00
8,Cali,Univalle,87695,60225,2,3.377911,-76.533811,2020-01-01 00:00:00,2024-12-31 23:00:00
9,Santiago de Cali,Base Aerea,13423,0,1,3.457128,-76.502303,2020-01-01 01:00:00,2023-12-09 23:00:00


estaciones sin coordenadas: 0


## 3. Calidad De Valores

No eliminamos agresivamente todavía. Primero dejamos trazabilidad: nulos, negativos y valores extremos por contaminante. Para el baseline, los negativos y nulos no deben entrar como observaciones válidas.

In [7]:
quality_summary = (
    df.assign(
        valor_nulo=df['valor'].isna(),
        valor_negativo=df['valor'] < 0,
        coord_nula=df[['lat', 'lon']].isna().any(axis=1),
    )
    .groupby('contaminante', dropna=False)
    .agg(
        n_obs=('valor', 'size'),
        n_nulos=('valor_nulo', 'sum'),
        n_negativos=('valor_negativo', 'sum'),
        n_coord_nula=('coord_nula', 'sum'),
        p01=('valor', lambda s: s.quantile(0.01)),
        p50=('valor', 'median'),
        p99=('valor', lambda s: s.quantile(0.99)),
    )
    .reset_index()
)
display(quality_summary)

,contaminante,n_obs,n_nulos,n_negativos,n_coord_nula,p01,p50,p99
0,NO2,178340,152081,0,0,0.000000,12.147438,46.555901
1,O3,340070,148813,0,0,0.058855,14.909903,135.483932
2,SO2,284736,152321,0,0,0.000000,2.853954,79.596520


## 4. Filtrar Contaminantes Objetivo

El PDF pide `NO2`, `SO2` y `O3`. Quitamos filas sin valor, sin coordenadas o con valores negativos para construir el panel diario base.

In [8]:
target_pollutants = ['NO2', 'SO2', 'O3']
clean_hourly = (
    df[df['contaminante'].isin(target_pollutants)]
    .dropna(subset=['valor', 'lat', 'lon', 'fecha_hora', 'date', 'estacion'])
    .query('valor >= 0')
    .copy()
)

print('hourly clean shape:', clean_hourly.shape)
display(clean_hourly.groupby('contaminante')['valor'].describe())

hourly clean shape: (349931, 15)


,count,mean,std,min,25%,50%,75%,max
contaminante,,,,,,,,
NO2,26259.0,14.058131,10.052477,0.0,6.957511,12.147438,18.973320,91.857945
O3,191257.0,28.550143,31.758247,0.0,6.964494,14.909903,39.670000,290.409591
SO2,132415.0,5.864049,13.248598,0.0,1.178238,2.853954,6.074471,841.968850


## 5. Agregación Horaria A Diaria

Para cada día, estación y contaminante guardamos media, mediana, mínimo, máximo, desviación y número de horas válidas. El campo principal para modelar será `valor_mean`.

In [9]:
daily_long = (
    clean_hourly.groupby(['date', 'municipio', 'estacion', 'lat', 'lon', 'contaminante', 'unidad'], dropna=False)
    .agg(
        valor_mean=('valor', 'mean'),
        valor_median=('valor', 'median'),
        valor_min=('valor', 'min'),
        valor_max=('valor', 'max'),
        valor_std=('valor', 'std'),
        n_horas_validas=('valor', 'count'),
    )
    .reset_index()
)
daily_long['date'] = pd.to_datetime(daily_long['date'])

display(daily_long.head())
print('daily long shape:', daily_long.shape)

,date,municipio,estacion,lat,lon,contaminante,unidad,valor_mean,valor_median,valor_min,valor_max,valor_std,n_horas_validas
0,2020-01-01,Cali,Base Aerea,3.457128,-76.502303,O3,ug/m3,32.609015,18.048830,1.373281,89.459416,33.529909,23
1,2020-01-01,Cali,Compartir,3.428260,-76.466584,O3,ug/m3,32.754020,21.305466,7.984645,82.043701,27.317550,23
2,2020-01-01,Cali,Ermita,3.455514,-76.530978,SO2,ug/m3,19.640715,18.249597,8.430947,40.636118,7.901713,23
3,2020-01-01,Cali,Pance,3.304517,-76.531252,O3,ug/m3,46.836542,28.838891,12.555708,123.791429,38.575281,23
4,2020-01-01,Cali,Univalle,3.377911,-76.533811,NO2,ug/m3,12.698694,10.586699,0.056412,33.508876,9.351164,19


daily long shape: (15653, 13)


In [10]:
daily_wide = (
    daily_long.pivot_table(
        index=['date', 'municipio', 'estacion', 'lat', 'lon'],
        columns='contaminante',
        values='valor_mean',
        aggfunc='mean',
    )
    .reset_index()
    .rename_axis(columns=None)
)
daily_wide = daily_wide.rename(columns={'NO2': 'no2_daily_mean', 'SO2': 'so2_daily_mean', 'O3': 'o3_daily_mean'})

display(daily_wide.head())
print('daily wide shape:', daily_wide.shape)

,date,municipio,estacion,lat,lon,no2_daily_mean,o3_daily_mean,so2_daily_mean
0,2020-01-01,Cali,Base Aerea,3.457128,-76.502303,NaN,32.609015,NaN
1,2020-01-01,Cali,Compartir,3.428260,-76.466584,NaN,32.754020,NaN
2,2020-01-01,Cali,Ermita,3.455514,-76.530978,NaN,NaN,19.640715
3,2020-01-01,Cali,Pance,3.304517,-76.531252,NaN,46.836542,NaN
4,2020-01-01,Cali,Univalle,3.377911,-76.533811,12.698694,32.733959,NaN


daily wide shape: (12280, 8)


In [11]:
daily_coverage = (
    daily_long.groupby(['date', 'contaminante'])
    .agg(
        n_estaciones=('estacion', 'nunique'),
        n_obs_diarias=('valor_mean', 'size'),
        valor_mean=('valor_mean', 'mean'),
    )
    .reset_index()
)
display(daily_coverage.groupby('contaminante')['n_estaciones'].describe())

,count,mean,std,min,25%,50%,75%,max
contaminante,,,,,,,,
NO2,1274.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
O3,1825.0,4.603836,1.291116,1.0,4.0,5.0,6.0,7.0
SO2,1803.0,3.221852,1.093635,1.0,2.0,3.0,4.0,5.0


## 6. Guardar Outputs

Estos archivos serán entrada directa para el baseline de Kriging y LOO-CV.

In [12]:
paths = {
    'daily_long': OUT_DIR / 'dagma_sisaire_daily_long.parquet',
    'daily_wide': OUT_DIR / 'dagma_sisaire_daily_wide.parquet',
    'station_summary': OUT_DIR / 'station_summary.csv',
    'contaminant_summary': OUT_DIR / 'contaminant_summary.csv',
    'quality_summary': OUT_DIR / 'quality_summary.csv',
    'daily_coverage': OUT_DIR / 'daily_coverage.csv',
}

daily_long.to_parquet(paths['daily_long'], index=False)
daily_wide.to_parquet(paths['daily_wide'], index=False)
station_summary.to_csv(paths['station_summary'], index=False)
contaminant_summary.to_csv(paths['contaminant_summary'], index=False)
quality_summary.to_csv(paths['quality_summary'], index=False)
daily_coverage.to_csv(paths['daily_coverage'], index=False)

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    'input_path': str(INPUT_PATH),
    'input_md5': md5_file(INPUT_PATH),
    'target_pollutants': target_pollutants,
    'hourly_clean_rows': int(len(clean_hourly)),
    'daily_long_rows': int(len(daily_long)),
    'daily_wide_rows': int(len(daily_wide)),
    'date_min': str(daily_long['date'].min()),
    'date_max': str(daily_long['date'].max()),
    'n_stations': int(daily_long['estacion'].nunique()),
    'coordinate_fill_summary': coord_fill_summary,
    'outputs': {k: str(v) for k, v in paths.items()},
    'output_md5': {k: md5_file(v) for k, v in paths.items()},
}
manifest_path = OUT_DIR / 'manifest_01_panel_dagma.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

manifest

{'input_path': '/workspace/geovision-cali-hf/data/DAGMA/processed/panel_dagma_sisaire_largo_2020_2024.parquet',
 'input_md5': '9895ea9786fd79e22fc98ea554f90b75',
 'target_pollutants': ['NO2', 'SO2', 'O3'],
 'hourly_clean_rows': 349931,
 'daily_long_rows': 15653,
 'daily_wide_rows': 12280,
 'date_min': '2020-01-01 00:00:00',
 'date_max': '2024-12-31 00:00:00',
 'n_stations': 9,
 'coordinate_fill_summary': {'coord_missing_before_fill': 614204,
  'coord_missing_after_fill': 0,
  'stations_with_coord_reference': 9},
 'outputs': {'daily_long': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/dagma_sisaire_daily_long.parquet',
  'daily_wide': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/dagma_sisaire_daily_wide.parquet',
  'station_summary': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/station_summary.csv',
  'contaminant_summary': '/workspace/geovision-cali-hf/outputs/situacion3/01_panel_dagma/contaminant_summary.csv',
  'quality_summary'

## Conclusión

Si este notebook corre completo, ya tenemos la verdad terreno diaria para empezar `03_baseline_kriging_loo_cv.ipynb`. Antes de eso, `02_grilla_cali_covariables.ipynb` debe preparar la grilla y enlazar covariables espaciales como Sentinel-2 con SCL.